### Build Chroma vector DB

In [ ]:
import os
import json
import re
import hashlib
from datetime import datetime
from pathlib import Path

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from orszaggyulesi_naplo import hunparl as hp

In [ ]:
PDF_DIR = Path("parllment/raw_pdf/2014-2018")
DB_DIR = Path("parllment/Chroma_db/chroma_db")
MEP_JSON = Path('orszaggyulesi_naplo/2014_2018_mep.json')

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/LaBSE")
vectordb = Chroma(
    persist_directory=str(DB_DIR),
    embedding_function=embeddings,
    collection_name="2014-2018"
)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=100, separators=["\n\n", "\n", ". ", " ", ""]
)

with open(MEP_JSON, 'r', encoding='utf-8') as f:
    mp_dict = json.load(f)

def capitalize_name(name: str) -> str:
    # Handles Hungarian accented characters and multipart names
    return re.sub(r"[A-ZÁÉÍÓÖŐÚÜŰa-záéíóöőúüű]+", lambda m: m.group().capitalize(), name).strip()

def convert_hungarian_date(date_str):
    try:
        months = {'január': '1', 'február': '2', 'március': '3', 'április': '4', 'május': '5', 'június': '6',
                  'július': '7', 'augusztus': '8', 'szeptember': '9', 'október': '10', 'november': '11', 'december': '12'}
        parts = date_str.replace('.', '').split()
        return f"{parts[0]}-{months[parts[1].lower()].zfill(2)}-{parts[2].zfill(2)}"
    except (IndexError, KeyError, AttributeError):
        return "1900-01-01"

In [ ]:
files = sorted(os.listdir(PDF_DIR))

for file in files:
    print(f"Processing: {file}")
    try:
        naplo = hp.OgyNaplo(PDF_DIR / file)
        
        datum = convert_hungarian_date(getattr(naplo, 'datum', ""))
        ciklus = str(getattr(naplo, 'ciklus', "Unknown")).split(".")[0]
        szam = getattr(naplo, 'szam', "Unknown")
        
        all_chunks = []
        
        for speaker_raw, speeches in naplo.beszedek.items():
            name = capitalize_name(speaker_raw)
            party = mp_dict.get(name, "Kormányzati tisztségviselő")
            
            texts = speeches if isinstance(speeches, list) else [speeches]
            
            for s_idx, text in enumerate(texts):
                if not text or len(text.strip()) < 10: continue
                    
                speech_uid = f"{file}_{name}_{s_idx}"
                speech_id = hashlib.md5(speech_uid.encode()).hexdigest()[:12]
                
                doc_chunks = text_splitter.split_documents([Document(page_content=text)])
                
                for c_idx, chunk in enumerate(doc_chunks):
                    c_uid = f"{speech_uid}_{c_idx}"
                    c_id = hashlib.md5(c_uid.encode()).hexdigest()[:14]
                    prev_uid = f"{speech_uid}_{c_idx-1}" if c_idx > 0 else None
                    next_uid = f"{speech_uid}_{c_idx+1}" if c_idx < len(doc_chunks)-1 else None                  
                    prev_uid = f"{speech_uid}_{c_idx-1}" if c_idx > 0 else None
                    next_uid = f"{speech_uid}_{c_idx+1}" if c_idx < len(doc_chunks)-1 else None
                    speech_preview = text[:150].strip() + "..."
                    datum_int = int(datum.replace("-", ""))
                    breadcrumb = f"{datum} | {name} ({party}) | Napló sz.: {szam}"
                    
                    chunk.metadata = {
                            "id": c_id,
                            "prev_id": hashlib.md5(prev_uid.encode()).hexdigest()[:14] if prev_uid else None,
                            "next_id": hashlib.md5(next_uid.encode()).hexdigest()[:14] if next_uid else None,
                            "speech_id": speech_id,
                            "chunk_seq": c_idx,
                            "source": file,
                            "ciklus": ciklus,
                            "szam": szam,
                            "datum": datum,
                            "datum_int": datum_int, 
                            "felszolalo": name,
                            "part": party,
                            "speech_preview": speech_preview,
                            "word_count": len(chunk.page_content.split()),
                            "breadcrumb" : breadcrumb,
                            "total_chunks": len(doc_chunks),
                            "token_estimate": len(chunk.page_content) // 4
                        }
                    
                    all_chunks.append(chunk)

        if all_chunks:
            vectordb.add_documents(
                documents=all_chunks, 
                ids=[c.metadata["id"] for c in all_chunks]
            )
            
    except Exception as e:
        print(f"\nError processing {file}: {e}")
        continue